In [0]:
-- Databricks notebook: 05_validate_silver_gold
-- Complete data quality report for 02_silver and 03_gold

USE CATALOG retail_catalog;

WITH 
row_counts AS (
    SELECT 'row_count' AS check_type, 'dimdate' AS object_name, CAST(COUNT(*) AS STRING) AS value FROM `02_silver`.silver_dimdate
    UNION ALL SELECT 'row_count', 'dimcustomer', CAST(COUNT(*) AS STRING) FROM `02_silver`.silver_dimcustomer
    UNION ALL SELECT 'row_count', 'dimproduct', CAST(COUNT(*) AS STRING) FROM `02_silver`.silver_dimproduct
    UNION ALL SELECT 'row_count', 'dimstore', CAST(COUNT(*) AS STRING) FROM `02_silver`.silver_dimstore
    UNION ALL SELECT 'row_count', 'dimpromotion', CAST(COUNT(*) AS STRING) FROM `02_silver`.silver_dimpromotion
    UNION ALL SELECT 'row_count', 'factsales', CAST(COUNT(*) AS STRING) FROM `02_silver`.silver_factsales
),
duplicates AS (
    SELECT 'duplicates' AS check_type, 'dimdate' AS object_name, CAST(COUNT(*) - COUNT(DISTINCT datekey) AS STRING) AS value FROM `02_silver`.silver_dimdate
    UNION ALL SELECT 'duplicates', 'dimcustomer', CAST(COUNT(*) - COUNT(DISTINCT customerid) AS STRING) FROM `02_silver`.silver_dimcustomer
    UNION ALL SELECT 'duplicates', 'dimproduct', CAST(COUNT(*) - COUNT(DISTINCT productid) AS STRING) FROM `02_silver`.silver_dimproduct
    UNION ALL SELECT 'duplicates', 'dimstore', CAST(COUNT(*) - COUNT(DISTINCT storeid) AS STRING) FROM `02_silver`.silver_dimstore
    UNION ALL SELECT 'duplicates', 'dimpromotion', CAST(COUNT(*) - COUNT(DISTINCT promoid) AS STRING) FROM `02_silver`.silver_dimpromotion
    UNION ALL SELECT 'duplicates', 'factsales', CAST(COUNT(*) - COUNT(DISTINCT salesid) AS STRING) FROM `02_silver`.silver_factsales
),
orphans AS (
    SELECT 'orphan_keys' AS check_type, 'missing datekey' AS object_name, CAST(COUNT(*) AS STRING) AS value FROM `02_silver`.silver_factsales f LEFT JOIN `02_silver`.silver_dimdate d ON f.datekey = d.datekey WHERE d.datekey IS NULL
    UNION ALL SELECT 'orphan_keys', 'missing productid', CAST(COUNT(*) AS STRING) FROM `02_silver`.silver_factsales f LEFT JOIN `02_silver`.silver_dimproduct p ON f.productid = p.productid WHERE p.productid IS NULL
    UNION ALL SELECT 'orphan_keys', 'missing customerid', CAST(COUNT(*) AS STRING) FROM `02_silver`.silver_factsales f LEFT JOIN `02_silver`.silver_dimcustomer c ON f.customerid = c.customerid WHERE c.customerid IS NULL
    UNION ALL SELECT 'orphan_keys', 'missing storeid', CAST(COUNT(*) AS STRING) FROM `02_silver`.silver_factsales f LEFT JOIN `02_silver`.silver_dimstore s ON f.storeid = s.storeid WHERE s.storeid IS NULL
    UNION ALL SELECT 'orphan_keys', 'missing promoid', CAST(COUNT(*) AS STRING) FROM `02_silver`.silver_factsales f LEFT JOIN `02_silver`.silver_dimpromotion pr ON f.promoid = pr.promoid WHERE pr.promoid IS NULL
),
range_checks AS (
    SELECT 'range_check' AS check_type, 'margin_pct out of [-0.10,0.30]' AS object_name, CAST(COUNT(*) AS STRING) AS value FROM `02_silver`.silver_dimproduct WHERE margin_pct < -0.10 OR margin_pct > 0.30
    UNION ALL SELECT 'range_check', 'tax_rate out of [0,1]', CAST(COUNT(*) AS STRING) FROM `02_silver`.silver_dimproduct WHERE tax_rate < 0 OR tax_rate > 1
    UNION ALL SELECT 'range_check', 'discount_pct out of [0,0.45]', CAST(COUNT(*) AS STRING) FROM `02_silver`.silver_dimpromotion WHERE discount_pct < 0 OR discount_pct > 0.45
),
hour_check AS (
    SELECT 'hour_validation' AS check_type, 'invalid hour (NULL or not 0-23)' AS object_name, CAST(COUNT(*) AS STRING) AS value FROM `02_silver`.silver_factsales WHERE hour IS NULL OR hour NOT BETWEEN 0 AND 23
),
return_reason AS (
    SELECT 'return_reason' AS check_type, 'non-return with reason != "No return"' AS object_name, CAST(COUNT(*) AS STRING) AS value FROM `02_silver`.silver_factsales WHERE isreturn = 0 AND returnreason != 'No return'
    UNION ALL SELECT 'return_reason', 'return with reason = "No return"', CAST(COUNT(*) AS STRING) FROM `02_silver`.silver_factsales WHERE isreturn = 1 AND returnreason = 'No return'
),
delivery_check AS (
    SELECT 'delivery_logic' AS check_type, 'In-Store with deliverydays != 0' AS object_name, CAST(COUNT(*) AS STRING) AS value FROM `02_silver`.silver_factsales WHERE channel = 'In-Store' AND deliverydays != 0
    UNION ALL SELECT 'delivery_logic', 'Online/Mobile App with deliverydays = 0 and not return', CAST(COUNT(*) AS STRING) FROM `02_silver`.silver_factsales WHERE channel IN ('Online','Mobile App') AND deliverydays = 0 AND isreturn = 0
),
gold_summary AS (
    SELECT 'gold_summary' AS check_type, 'rows in vw_001_product_category_margin' AS object_name, CAST(COUNT(*) AS STRING) AS value FROM `03_gold`.vw_001_product_category_margin
    UNION ALL SELECT 'gold_summary', 'total revenue', CAST(SUM(total_revenue) AS STRING) FROM `03_gold`.vw_001_product_category_margin
)
SELECT * FROM row_counts
UNION ALL SELECT * FROM duplicates
UNION ALL SELECT * FROM orphans
UNION ALL SELECT * FROM range_checks
UNION ALL SELECT * FROM hour_check
UNION ALL SELECT * FROM return_reason
UNION ALL SELECT * FROM delivery_check
UNION ALL SELECT * FROM gold_summary
ORDER BY check_type, object_name;

check_type,object_name,value
delivery_logic,In-Store with deliverydays != 0,0
delivery_logic,Online/Mobile App with deliverydays = 0 and not return,0
duplicates,dimcustomer,0
duplicates,dimdate,0
duplicates,dimproduct,0
duplicates,dimpromotion,0
duplicates,dimstore,0
duplicates,factsales,0
gold_summary,rows in vw_001_product_category_margin,2000
gold_summary,total revenue,1499088531.28
